<a href="https://colab.research.google.com/github/kavipriyario/GroupDNA/blob/main/Kavipriya_GroupDNA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

KavipriyaR_Aug2026_DataScience

# GROUPDNA — WhatsApp Chat Analyzer

This project analyzes the Hostel Bois WhatsApp chat using only Python fundamentals and NumPy.

Allowed:
- Python fundamentals
- Lists, tuples, sets and dictionaries
- NumPy
- open(), readlines()
- datetime.strptime() and timedelta
- Basic string methods
- List/dictionary comprehensions
- sorted()

Not allowed:
- pandas
- matplotlib
- seaborn
- plotly
- collections.Counter
- collections.defaultdict
- re
- AI/ML libraries
- WhatsApp analyzer libraries
- External datasets

In [240]:
import numpy as np
from datetime import datetime, timedelta

## Step 1 — Read the WhatsApp chat file

The dataset used for this project is hostel_bois.txt.

In [241]:
file_name = "hostel_bois.txt"

with open(file_name, "r", encoding="utf-8") as file:
    lines = file.readlines()

print("Total physical lines:", len(lines))

Total physical lines: 3178


## Step 2 — Parse timestamps

WhatsApp exports contain a timestamp at the beginning of every new message.

We try the common WhatsApp timestamp formats without using regular expressions.

In [242]:
def parse_timestamp(text):
    text = text.strip()

    text = text.replace("\ufeff", "")
    text = text.replace("\u200e", "")
    text = text.replace("\u200f", "")
    text = text.replace("\u202f", " ")
    text = text.replace("\xa0", " ")

    formats = [
        "%d/%m/%y, %I:%M %p",
        "%d/%m/%Y, %I:%M %p",
        "%d/%m/%y, %H:%M",
        "%d/%m/%Y, %H:%M",
        "%d/%m/%y, %I:%M:%S %p",
        "%d/%m/%Y, %I:%M:%S %p",
        "%d/%m/%y, %H:%M:%S",
        "%d/%m/%Y, %H:%M:%S"
    ]

    for fmt in formats:
        try:
            return datetime.strptime(text, fmt)
        except:
            pass

    return None

## Step 3 — Clean and parse messages

The parser handles:
- Normal messages
- Multiline messages
- System messages
- Media messages
- Deleted messages

Only real user messages are stored.

In [243]:
def clean_line(line):
    line = line.strip()

    line = line.replace("\ufeff", "")
    line = line.replace("\u200e", "")
    line = line.replace("\u200f", "")
    line = line.replace("\u2066", "")
    line = line.replace("\u2067", "")
    line = line.replace("\u2068", "")
    line = line.replace("\u2069", "")

    return line


def parse_chat(lines):
    parsed = []
    current = None

    for raw_line in lines:

        line = clean_line(raw_line)

        if line == "":
            continue

        timestamp = None
        rest = ""

        parts = line.split(" - ", 1)

        if len(parts) == 2:
            timestamp = parse_timestamp(parts[0])

            if timestamp is not None:
                rest = parts[1]

        # New WhatsApp message
        if timestamp is not None:

            if current is not None:
                parsed.append(current)

            sender_and_text = rest.split(": ", 1)

            # User message
            if len(sender_and_text) == 2:

                sender = sender_and_text[0].strip()
                text = sender_and_text[1].strip()

                current = {
                    "timestamp": timestamp,
                    "sender": sender,
                    "text": text
                }

            # System message
            else:
                current = None

        # Multiline continuation
        else:

            if current is not None:
                current["text"] += " " + line

    # Save final message
    if current is not None:
        parsed.append(current)

    # Remove media and deleted messages
    messages = []

    for message in parsed:

        text = message["text"].lower()

        if "<media omitted>" in text:
            continue

        if "this message was deleted" in text:
            continue

        if "you deleted this message" in text:
            continue

        messages.append(message)

    return messages

## Step 4 — Parse the complete dataset

The assignment checkpoint is 3,174 real messages.

In [244]:
messages = parse_chat(lines)

print("Messages parsed:", len(messages))

Messages parsed: 3127


## Step 5 — Verify the parser

We print the first five and last five messages to make sure the parser is working correctly.

In [245]:
print("FIRST 5 MESSAGES")
print("=" * 60)

for message in messages[:5]:
    print(message)

print()
print("LAST 5 MESSAGES")
print("=" * 60)

for message in messages[-5:]:
    print(message)

FIRST 5 MESSAGES
{'timestamp': datetime.datetime(2024, 4, 1, 1, 17), 'sender': 'Rahul', 'text': 'scene fix'}
{'timestamp': datetime.datetime(2024, 4, 1, 1, 17), 'sender': 'Rahul', 'text': 'haan'}
{'timestamp': datetime.datetime(2024, 4, 1, 1, 18), 'sender': 'Rahul', 'text': 'kya scene'}
{'timestamp': datetime.datetime(2024, 4, 1, 2, 13), 'sender': 'Rahul', 'text': 'abhi free hai?'}
{'timestamp': datetime.datetime(2024, 4, 1, 2, 13), 'sender': 'Rahul', 'text': 'abey'}

LAST 5 MESSAGES
{'timestamp': datetime.datetime(2024, 5, 30, 19, 14), 'sender': 'Priya', 'text': 'Take care everyone'}
{'timestamp': datetime.datetime(2024, 5, 30, 19, 28), 'sender': 'Priya', 'text': 'Karan that sounds tough, take care'}
{'timestamp': datetime.datetime(2024, 5, 30, 21, 17), 'sender': 'Aman', 'text': 'the existential dread is back'}
{'timestamp': datetime.datetime(2024, 5, 30, 21, 30), 'sender': 'Karan', 'text': 'Long day guys, woke up at six for that placement workshop which started at eight by the way cl

## Step 6 — Find all group members

In [246]:
people = []

for message in messages:

    sender = message["sender"]

    if sender not in people:
        people.append(sender)

print("Number of members:", len(people))
print("Members:", people)

Number of members: 6
Members: ['Rahul', 'Priya', 'Karan', 'Neha', 'Aman', 'Vikas']


# Feature 4 - Activity Heatmap

In [247]:
activity = np.zeros((len(people), 24), dtype=int)

for message in messages:
    text = message["text"]
    if text == "<Media omitted>" or text == "This message was deleted":
        continue

    person = message["sender"]
    hour = message["timestamp"].hour

    if person in people:
        row = people.index(person)
        activity[row][hour] += 1

print("Activity matrix")
print(activity)

Activity matrix
[[  3  15  17  17  22   9  17  17  23  17  25  15  57  48  45  53  71  48
  102  74  40  92  60  53]
 [  0   0   0   0   0   0  13  20  46  64  61  61  57  48  44  28  32  40
   38  59  42  32  18   9]
 [  0   0   0   0   0   0   0   4  11  16  19  16  36  22  32  26  27  27
   24  32  23  14   9   7]
 [  0   0   0   0   0  19   3  13  35  51  52  21  39  36  26   9  36  47
   62  49  44  26  26  30]
 [ 53  67  66  60  87   0   0   0   0   0   0   0   0   0  14  11  18   5
   16   8  12  11   0  56]
 [  0   0   0   0   0   0   0   1   2   1   1   0   1   2   0   1   1   3
    2   2   1   1   1   2]]


## Step 7 — Messages per person

We count messages manually using a dictionary instead of collections.Counter.

In [248]:
message_counts = {}

for person in people:
    message_counts[person] = 0

for message in messages:
    sender = message["sender"]
    message_counts[sender] += 1

sorted_people = sorted(
    message_counts.items(),
    key=lambda item: item[1],
    reverse=True
)

for person, count in sorted_people:
    print(person, count)

Rahul 940
Priya 712
Neha 624
Aman 484
Karan 345
Vikas 22


## Step 8 — Text bar function

The bar length is proportional to the largest value.

A very small non-zero value such as Vikas's 24 messages is displayed as "." instead of incorrectly forcing one full block.

In [249]:
def make_bar(value, maximum, width=20):

    if maximum <= 0:
        return "."

    length = int((value / maximum) * width)

    if length <= 0:
        return "."

    return "█" * length

## Step 9 — Messages Per Person Report

In [250]:
print("MESSAGES PER PERSON")

maximum_messages = sorted_people[0][1]

for person, count in sorted_people:

    percentage = (
        count / len(messages)
    ) * 100

    bar = make_bar(
        count,
        maximum_messages,
        20
    )

    print(
        f"{person:<7} "
        f"{bar:<20} "
        f"{count:>4} "
        f"({percentage:.1f}%)"
    )

MESSAGES PER PERSON
Rahul   ████████████████████  940 (30.1%)
Priya   ███████████████       712 (22.8%)
Neha    █████████████         624 (20.0%)
Aman    ██████████            484 (15.5%)
Karan   ███████               345 (11.0%)
Vikas   .                      22 (0.7%)


## Step 10 — Date Range

In [251]:
start_timestamp = messages[0]["timestamp"]
end_timestamp = messages[-1]["timestamp"]

number_of_days = (
    end_timestamp - start_timestamp
).days + 1

print("Start:", start_timestamp)
print("End:", end_timestamp)
print("Number of days:", number_of_days)

Start: 2024-04-01 01:17:00
End: 2024-05-30 23:31:00
Number of days: 60


## Step 11 — Busiest Day

In [252]:
day_counts = {}

for message in messages:

    timestamp = message["timestamp"]

    day = (
        timestamp.year,
        timestamp.month,
        timestamp.day
    )

    day_counts[day] = day_counts.get(day, 0) + 1

busiest_day = max(
    day_counts,
    key=day_counts.get
)

busiest_day_count = day_counts[busiest_day]

print(
    "Busiest day:",
    busiest_day,
    busiest_day_count,
    "messages"
)

Busiest day: (2024, 5, 4) 74 messages


## Step 12 — Busiest Hour

In [253]:
hour_counts = {}

for hour in range(24):
    hour_counts[hour] = 0

for message in messages:

    hour = message["timestamp"].hour

    hour_counts[hour] += 1

busiest_hour = max(
    hour_counts,
    key=hour_counts.get
)

print(
    f"Busiest hour: "
    f"{busiest_hour:02d}:00 - "
    f"{(busiest_hour + 1) % 24:02d}:00"
)

Busiest hour: 18:00 - 19:00


In [254]:
active_dates = {}
for person in people:
    active_dates[person] = set()

for message in messages:
    person = message["sender"]
    date = message["timestamp"].date()
    active_dates[person].add(date)

silent_streaks = {}
for person in people:
    current = start_timestamp.date()
    longest = 0
    streak = 0
    while current <= end_timestamp.date():
        if current not in active_dates[person]:
            streak += 1
            if streak > longest:
                longest = streak
        else:
            streak = 0
        current += timedelta(days=1)
    silent_streaks[person] = longest

print("LONGEST SILENT STREAKS")
for person in people:
    print(person, ":", silent_streaks[person], "days")

LONGEST SILENT STREAKS
Rahul : 0 days
Priya : 0 days
Karan : 0 days
Neha : 0 days
Aman : 0 days
Vikas : 11 days


## Step 13 — Word Frequency

We create our own dictionary-based word counter.


In [255]:
stop_words = {
    "a", "an", "the",
    "is", "am", "are",
    "was", "were",
    "be", "been", "being",
    "i", "me", "my",
    "we", "us", "our",
    "you", "your",
    "he", "him", "his",
    "she", "her",
    "it", "its",
    "they", "them", "their",
    "this", "that",
    "these", "those",
    "to", "of", "in",
    "on", "at", "for",
    "from", "with", "by",
    "and", "or", "but",
    "if", "then", "than",
    "so", "as",
    "do", "does", "did",
    "have", "has", "had",
    "will", "would",
    "can", "could",
    "should",
    "not", "no",
    "very", "just"
}

def tokenize(text):
    text = text.lower()
    symbols = [
        ".", ",", "!", "?",
        ":", ";", "(", ")",
        "[", "]", "{", "}",
        "'", '"', "`",
        "-", "_", "/",
        "\\", "@", "#",
        "$", "%", "^",
        "&", "*", "+",
        "=", "|", "<",
        ">", "~"
    ]
    for symbol in symbols:
        text = text.replace(symbol, " ")
    raw_words = text.split()
    words = []
    for word in raw_words:
        if word.isalpha() and word not in stop_words:
            words.append(word)
    return words

## Step 14 — Group Favourite Words

In [256]:
word_frequency = {}
for message in messages:
    words = tokenize(
        message["text"]
    )
    for word in words:
        word_frequency[word] = (
            word_frequency.get(word, 0) + 1
        )

top_words = sorted(
    word_frequency.items(),
    key=lambda item: item[1],
    reverse=True
)

print("TOP 20 WORDS")
for word, count in top_words[:20]:
    print(word, count)

TOP 20 WORDS
how 321
guys 318
today 292
about 274
hai 268
s 226
everyone 203
which 202
telling 179
up 172
bhai 160
anyone 157
one 157
started 150
scene 145
entire 145
please 141
yaar 139
kya 133
t 130


## Step 15 — Favourite Words Bar Chart

In [257]:
print("THIS GROUP'S FAVOURITE WORDS")

maximum_word_count = top_words[0][1]

for word, count in top_words[:5]:

    bar = make_bar(
        count,
        maximum_word_count,
        20
    )

    print(
        f"{word:<8} "
        f"{bar:<20} "
        f"{count}"
    )

THIS GROUP'S FAVOURITE WORDS
how      ████████████████████ 321
guys     ███████████████████  318
today    ██████████████████   292
about    █████████████████    274
hai      ████████████████     268


## Step 16 — NumPy Activity Heatmap

The heatmap has:
- One row for each member
- 24 columns for the 24 hours of the day

In [258]:
person_index = {}

for i in range(len(people)):
    person_index[people[i]] = i

heatmap = np.zeros(
    (len(people), 24),
    dtype=int
)

for message in messages:

    person = message["sender"]
    hour = message["timestamp"].hour

    row = person_index[person]

    heatmap[row][hour] += 1

print(heatmap)

[[  3  15  17  17  22   9  17  17  23  17  25  15  57  48  45  53  71  48
  102  74  40  92  60  53]
 [  0   0   0   0   0   0  13  20  46  64  61  61  57  48  44  28  32  40
   38  59  42  32  18   9]
 [  0   0   0   0   0   0   0   4  11  16  19  16  36  22  32  26  27  27
   24  32  23  14   9   7]
 [  0   0   0   0   0  19   3  13  35  51  52  21  39  36  26   9  36  47
   62  49  44  26  26  30]
 [ 53  67  66  60  87   0   0   0   0   0   0   0   0   0  14  11  18   5
   16   8  12  11   0  56]
 [  0   0   0   0   0   0   0   1   2   1   1   0   1   2   0   1   1   3
    2   2   1   1   1   2]]


## Step 17 — Heatmap Shading

In [259]:
def heat_character(value, maximum):

    if value == 0:
        return "."

    ratio = value / maximum

    if ratio < 0.25:
        return "░"

    elif ratio < 0.50:
        return "▒"

    elif ratio < 0.75:
        return "▓"

    else:
        return "█"

## Step 18 — Print Activity Heatmap

In [260]:
print(
    "ACTIVITY HEATMAP "
    "(hour of day, columns 00 to 23)"
)

print("       ", end="")

for hour in range(24):
    print(f"{hour:02d} ", end="")

print()

maximum_heat = np.max(heatmap)

for i in range(len(people)):

    person = people[i]

    print(
        f"{person:<7}",
        end=""
    )

    for hour in range(24):

        character = heat_character(
            heatmap[i][hour],
            maximum_heat
        )

        print(
            f"{character}  ",
            end=""
        )

    if person == "Aman":
        print("<- NIGHT OWL")

    else:
        print()

ACTIVITY HEATMAP (hour of day, columns 00 to 23)
       00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23 
Rahul  ░  ░  ░  ░  ░  ░  ░  ░  ░  ░  ░  ░  ▓  ▒  ▒  ▓  ▓  ▒  █  ▓  ▒  █  ▓  ▓  
Priya  .  .  .  .  .  .  ░  ░  ▒  ▓  ▓  ▓  ▓  ▒  ▒  ▒  ▒  ▒  ▒  ▓  ▒  ▒  ░  ░  
Karan  .  .  .  .  .  .  .  ░  ░  ░  ░  ░  ▒  ░  ▒  ▒  ▒  ▒  ░  ▒  ░  ░  ░  ░  
Neha   .  .  .  .  .  ░  ░  ░  ▒  ▓  ▓  ░  ▒  ▒  ▒  ░  ▒  ▒  ▓  ▒  ▒  ▒  ▒  ▒  
Aman   ▓  ▓  ▓  ▓  █  .  .  .  .  .  .  .  .  .  ░  ░  ░  ░  ░  ░  ░  ░  .  ▓  <- NIGHT OWL
Vikas  .  .  .  .  .  .  .  ░  ░  ░  ░  .  ░  ░  .  ░  ░  ░  ░  ░  ░  ░  ░  ░  


## Step 19 — Response Time Analysis

A response is calculated when a person sends a message after another person.

Messages from the same person are skipped when looking for the previous sender.

In [261]:
response_times = {}

for person in people:
    response_times[person] = []

for i in range(1, len(messages)):

    current = messages[i]

    j = i - 1

    while j >= 0:

        previous = messages[j]

        if previous["sender"] != current["sender"]:

            gap = (
                current["timestamp"]
                - previous["timestamp"]
            )

            if gap.days >= 0:

                seconds = (
                    gap.days * 24 * 60 * 60
                    + gap.seconds
                )

                response_times[
                    current["sender"]
                ].append(seconds)

            break

        j -= 1

## Step 20 — Average Response Time

In [262]:
average_response_minutes = {}

for person in people:

    gaps = response_times[person]

    if len(gaps) == 0:

        average_response_minutes[person] = 0

    else:

        total_seconds = 0

        for gap in gaps:
            total_seconds += gap

        average_response_minutes[person] = (
            total_seconds
            / len(gaps)
            / 60
        )

for person in people:

    print(
        person,
        round(
            average_response_minutes[person],
            2
        ),
        "minutes"
    )

Rahul 39.72 minutes
Priya 69.12 minutes
Karan 44.26 minutes
Neha 49.16 minutes
Aman 157.73 minutes
Vikas 35.32 minutes


## Step 21 — Fastest and Slowest Repliers

In [263]:
people_with_responses = []

for person in people:

    if len(response_times[person]) > 0:
        people_with_responses.append(person)

fastest = min(
    people_with_responses,
    key=lambda person:
        average_response_minutes[person]
)

slowest = max(
    people_with_responses,
    key=lambda person:
        average_response_minutes[person]
)

print(
    "Fastest replier:",
    fastest
)

print(
    "Slowest replier:",
    slowest
)

Fastest replier: Vikas
Slowest replier: Aman


## Step 22 — Silent Streaks

We first create a set of dates on which each person sent at least one message.

In [264]:
active_dates = {}

for person in people:
    active_dates[person] = set()

for message in messages:

    timestamp = message["timestamp"]

    day = (
        timestamp.year,
        timestamp.month,
        timestamp.day
    )

    active_dates[
        message["sender"]
    ].add(day)

In [265]:
def get_day(timestamp):

    return (
        timestamp.year,
        timestamp.month,
        timestamp.day
    )


def longest_silent_streak(person):

    longest = 0
    current = 0

    total_days = (
        end_timestamp - start_timestamp
    ).days + 1

    for i in range(total_days):

        day = (
            start_timestamp
            + timedelta(days=i)
        )

        day_tuple = get_day(day)

        if day_tuple not in active_dates[person]:

            current += 1

            if current > longest:
                longest = current

        else:
            current = 0

    return longest

In [266]:
silent_streaks = {}

for person in people:

    silent_streaks[person] = (
        longest_silent_streak(person)
    )

sorted_silence = sorted(
    silent_streaks.items(),
    key=lambda item: item[1],
    reverse=True
)

print("LONGEST SILENT STREAKS")

for person, days in sorted_silence:

    print(
        f"{person:<8}: {days} days"
    )

LONGEST SILENT STREAKS
Vikas   : 11 days
Rahul   : 0 days
Priya   : 0 days
Karan   : 0 days
Neha    : 0 days
Aman    : 0 days


## Step 23 — Prepare Messages by Person

In [267]:
messages_by_person = {}

for person in people:
    messages_by_person[person] = []

for message in messages:

    messages_by_person[
        message["sender"]
    ].append(message)

## Step 24 — Archetype: The Spammer

This measures the average number of consecutive messages sent by the same person.

In [268]:
def spammer_score(person):

    streaks = []
    current = 0

    for message in messages:

        if message["sender"] == person:

            current += 1

        else:

            if current > 0:
                streaks.append(current)

            current = 0

    if current > 0:
        streaks.append(current)

    if len(streaks) == 0:
        return 0

    total = 0

    for streak in streaks:
        total += streak

    return total / len(streaks)

## Step 25 — Archetype: The Group Mom

In [269]:
caring_words = {
    "care",
    "take",
    "food",
    "eat",
    "eating",
    "sleep",
    "slept",
    "health",
    "home",
    "safe",
    "reach",
    "reached",
    "water",
    "medicine",
    "rest",
    "please",
    "help",
    "call"
}


def group_mom_score(person):

    score = 0

    for message in messages_by_person[person]:

        words = tokenize(
            message["text"]
        )

        for word in words:

            if word in caring_words:
                score += 1

    return score

## Step 26 — Archetype: The Night Owl

In [270]:
def night_owl_score(person):

    person_messages = messages_by_person[person]

    if len(person_messages) == 0:
        return 0

    night_messages = 0

    for message in person_messages:

        hour = message["timestamp"].hour

        if hour >= 23 or hour <= 4:
            night_messages += 1

    return (
        night_messages
        / len(person_messages)
        * 100
    )

## Step 27 — Archetype: The Storyteller

In [271]:
def storyteller_score(person):

    person_messages = messages_by_person[person]

    if len(person_messages) == 0:
        return 0

    total_words = 0

    for message in person_messages:

        total_words += len(
            message["text"].split()
        )

    return (
        total_words
        / len(person_messages)
    )

## Step 28 — Archetype: The Drama Queen

A message is considered ALL-CAPS when it contains at least three alphabetic characters and all of those alphabetic characters are uppercase.

In [272]:
def is_all_caps(text):

    letters = 0
    uppercase_letters = 0

    for character in text:

        if character.isalpha():

            letters += 1

            if character.isupper():
                uppercase_letters += 1

    if letters < 3:
        return False

    return letters == uppercase_letters


def drama_queen_score(person):

    person_messages = messages_by_person[person]

    if len(person_messages) == 0:
        return 0

    caps_messages = 0

    for message in person_messages:

        if is_all_caps(
            message["text"]
        ):
            caps_messages += 1

    return (
        caps_messages
        / len(person_messages)
        * 100
    )

## Step 29 — Archetype: The Ghost

In [273]:
def ghost_score(person):

    total_days = (
        end_timestamp - start_timestamp
    ).days + 1

    active_day_count = len(
        active_dates[person]
    )

    return total_days - active_day_count

## Step 30 — Display Archetype Metrics

These values are printed separately so that the classification logic can be checked against the Section 7 requirements.

In [274]:
print("ARCHETYPE METRICS")
print("=" * 60)

for person in people:

    print()
    print(person)
    print("-" * 40)

    print(
        "Average consecutive messages:",
        round(
            spammer_score(person),
            2
        )
    )

    print(
        "Caring keyword score:",
        group_mom_score(person)
    )

    print(
        "Night messages percentage:",
        round(
            night_owl_score(person),
            1
        ),
        "%"
    )

    print(
        "Average words/message:",
        round(
            storyteller_score(person),
            1
        )
    )

    print(
        "ALL-CAPS percentage:",
        round(
            drama_queen_score(person),
            1
        ),
        "%"
    )

    print(
        "Silent days:",
        ghost_score(person)
    )

ARCHETYPE METRICS

Rahul
----------------------------------------
Average consecutive messages: 4.48
Caring keyword score: 0
Night messages percentage: 13.5 %
Average words/message: 2.6
ALL-CAPS percentage: 0.0 %
Silent days: 0

Priya
----------------------------------------
Average consecutive messages: 1.67
Caring keyword score: 583
Night messages percentage: 1.3 %
Average words/message: 5.0
ALL-CAPS percentage: 0.0 %
Silent days: 0

Karan
----------------------------------------
Average consecutive messages: 1.24
Caring keyword score: 35
Night messages percentage: 2.0 %
Average words/message: 57.0
ALL-CAPS percentage: 0.0 %
Silent days: 0

Neha
----------------------------------------
Average consecutive messages: 2.53
Caring keyword score: 17
Night messages percentage: 4.8 %
Average words/message: 5.3
ALL-CAPS percentage: 63.3 %
Silent days: 0

Aman
----------------------------------------
Average consecutive messages: 2.77
Caring keyword score: 71
Night messages percentage: 80.4 %

## Step 31 — Final Validation

Before submitting, this cell checks the major target values from the assignment.

In [275]:
print("=" * 60)
print("GROUPDNA VALIDATION")
print("=" * 60)

print(
    "Total messages:",
    len(messages),
    "| Target: 3174"
)

print(
    "Total members:",
    len(people),
    "| Target: 6"
)

print()

print("MESSAGE COUNTS")

for person, count in sorted_people:

    print(
        f"{person:<8} "
        f"{count}"
    )

print()

print("TOP 5 WORDS")

for word, count in top_words[:5]:

    print(
        f"{word:<8} "
        f"{count}"
    )

print()

print("LONGEST SILENT STREAKS")

for person, days in sorted_silence:

    print(
        f"{person:<8} "
        f"{days} days"
    )

print("=" * 60)

GROUPDNA VALIDATION
Total messages: 3127 | Target: 3174
Total members: 6 | Target: 6

MESSAGE COUNTS
Rahul    940
Priya    712
Neha     624
Aman     484
Karan    345
Vikas    22

TOP 5 WORDS
how      321
guys     318
today    292
about    274
hai      268

LONGEST SILENT STREAKS
Vikas    11 days
Rahul    0 days
Priya    0 days
Karan    0 days
Neha     0 days
Aman     0 days


# Final GroupDNA Report

The following cell produces the polished final output for the submission.

In [276]:
print()
print("=" * 70)
print('GROUPDNA REPORT — "Hostel Bois 4ever"')
print(
    f"{number_of_days} days • "
    f"{len(messages):,} messages • "
    f"{len(people)} members"
)
print("=" * 70)

print()

print(
    f"Period : "
    f"{start_timestamp.day:02d}/"
    f"{start_timestamp.month:02d}/"
    f"{start_timestamp.year} "
    f"to "
    f"{end_timestamp.day:02d}/"
    f"{end_timestamp.month:02d}/"
    f"{end_timestamp.year}"
)

print(
    f"Busiest day : "
    f"{busiest_day[2]:02d}/"
    f"{busiest_day[1]:02d}/"
    f"{busiest_day[0]} "
    f"({busiest_day_count} messages)"
)

print(
    f"Busiest hour : "
    f"{busiest_hour:02d}:00 - "
    f"{(busiest_hour + 1) % 24:02d}:00"
)

print()
print("MESSAGES PER PERSON")

maximum_messages = sorted_people[0][1]

for person, count in sorted_people:

    percentage = (
        count / len(messages)
    ) * 100

    bar = make_bar(
        count,
        maximum_messages,
        20
    )

    print(
        f"{person:<7} "
        f"{bar:<20} "
        f"{count:>4} "
        f"({percentage:.1f}%)"
    )

print()
print("ACTIVITY HEATMAP")
print("(hour of day, columns 00 to 23)")

print("       ", end="")

for hour in range(24):
    print(f"{hour:02d} ", end="")

print()

maximum_heat = np.max(heatmap)

for i in range(len(people)):

    person = people[i]

    print(
        f"{person:<7}",
        end=""
    )

    for hour in range(24):

        character = heat_character(
            heatmap[i][hour],
            maximum_heat
        )

        print(
            f"{character}  ",
            end=""
        )

    if person == "Aman":
        print("<- NIGHT OWL")
    else:
        print()

print()
print("THIS GROUP'S FAVOURITE WORDS")

maximum_word_count = top_words[0][1]

for word, count in top_words[:5]:

    bar = make_bar(
        count,
        maximum_word_count,
        20
    )

    print(
        f"{word:<8} "
        f"{bar:<20} "
        f"{count}"
    )

print()
print("RESPONSE PATTERNS")

fast_minutes = average_response_minutes[fastest]
slow_minutes = average_response_minutes[slowest]

print(
    f"Fastest replier : "
    f"{fastest} "
    f"(avg {fast_minutes:.1f} minutes)"
)

if slow_minutes >= 60:

    print(
        f"Slowest replier : "
        f"{slowest} "
        f"(avg {slow_minutes / 60:.1f} hours)"
    )

else:

    print(
        f"Slowest replier : "
        f"{slowest} "
        f"(avg {slow_minutes:.1f} minutes)"
    )

print()
print("LONGEST SILENT STREAKS")

for person, days in sorted_silence[:4]:

    print(
        f"{person:<8}: "
        f"{days} days"
    )

print()
print("PERSONALITY ARCHETYPE METRICS")

for person in people:

    print()
    print(person)

    print(
        "  Spammer:",
        round(
            spammer_score(person),
            1
        )
    )

    print(
        "  Group Mom:",
        group_mom_score(person)
    )

    print(
        "  Night Owl:",
        round(
            night_owl_score(person),
            1
        ),
        "%"
    )

    print(
        "  Storyteller:",
        round(
            storyteller_score(person),
            1
        ),
        "words/message"
    )

    print(
        "  Drama Queen:",
        round(
            drama_queen_score(person),
            1
        ),
        "%"
    )

    print(
        "  Ghost:",
        ghost_score(person),
        "silent days"
    )

print()
print("=" * 70)
print("Generated by GroupDNA • Built with Python + NumPy")
print("=" * 70)


GROUPDNA REPORT — "Hostel Bois 4ever"
60 days • 3,127 messages • 6 members

Period : 01/04/2024 to 30/05/2024
Busiest day : 04/05/2024 (74 messages)
Busiest hour : 18:00 - 19:00

MESSAGES PER PERSON
Rahul   ████████████████████  940 (30.1%)
Priya   ███████████████       712 (22.8%)
Neha    █████████████         624 (20.0%)
Aman    ██████████            484 (15.5%)
Karan   ███████               345 (11.0%)
Vikas   .                      22 (0.7%)

ACTIVITY HEATMAP
(hour of day, columns 00 to 23)
       00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23 
Rahul  ░  ░  ░  ░  ░  ░  ░  ░  ░  ░  ░  ░  ▓  ▒  ▒  ▓  ▓  ▒  █  ▓  ▒  █  ▓  ▓  
Priya  .  .  .  .  .  .  ░  ░  ▒  ▓  ▓  ▓  ▓  ▒  ▒  ▒  ▒  ▒  ▒  ▓  ▒  ▒  ░  ░  
Karan  .  .  .  .  .  .  .  ░  ░  ░  ░  ░  ▒  ░  ▒  ▒  ▒  ▒  ░  ▒  ░  ░  ░  ░  
Neha   .  .  .  .  .  ░  ░  ░  ▒  ▓  ▓  ░  ▒  ▒  ▒  ░  ▒  ▒  ▓  ▒  ▒  ▒  ▒  ▒  
Aman   ▓  ▓  ▓  ▓  █  .  .  .  .  .  .  .  .  .  ░  ░  ░  ░  ░  ░  ░  ░  .  ▓  <- NIGHT OWL
Vikas  

# Reflection



This project was a good learning experience for me because I got to use Python to analyze a real WhatsApp chat. At first, parsing the chat and organizing the messages was a little difficult, but I understood it better after working with lists and dictionaries.

I learned how to count messages, find the most common words, calculate response times, and find which days people were active. I also learned how to use NumPy to create the activity heatmap.

One of the main things I learned was that I can create useful analysis without using advanced libraries like pandas. I had to use basic Python concepts such as loops, conditions, functions, and dictionaries, which helped me understand them better.

Overall, this project improved my Python skills and gave me more confidence in working with data. If I had more time, I would improve the visualizations and add more interesting features to the analysis.